In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import joblib

In [ ]:
RAW = Path("../Dataset/Raw")
PROCESSED = Path("../Dataset/Processed")

df_meta = pd.read_csv(RAW / "games_detailed_info.csv", low_memory=False)
df_meta = df_meta.rename(columns={"primary":"name","boardgamemechanic":"mechanic","boardgamecategory":"category"})
df_meta = df_meta[["id","name","description"]].drop_duplicates(subset="id").set_index("id")

df_reviews = pd.read_csv(PROCESSED / "reviews_with_sentiment.csv", low_memory=False)

In [ ]:
df_meta["description"] = df_meta["description"].fillna("")

tf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)
desc_tfidf = tf.fit_transform(df_meta["description"].values)
print("TF-IDF shape:", desc_tfidf.shape)
joblib.dump(tf, "../Dataset/Processed/tfidf_vec.joblib")
joblib.dump(desc_tfidf, "../Dataset/Processed/desc_tfidf.npz")

In [ ]:
mech = pd.read_csv(RAW / "mechanics.csv", low_memory=False).set_index("BGGId")
themes = pd.read_csv(RAW / "themes.csv", low_memory=False).set_index("BGGId")
subc = pd.read_csv(RAW / "subcategories.csv", low_memory=False).set_index("BGGId")

for df in [mech, themes, subc]:
    df.index = df.index.astype(int)

ids = df_meta.index.astype(int)
mech = mech.reindex(ids).fillna(0)
themes = themes.reindex(ids).fillna(0)
subc = subc.reindex(ids).fillna(0)

content_matrix = np.hstack([mech.values, themes.values, subc.values])
content_matrix = normalize(content_matrix, norm='l2', axis=1)
print("Content matrix shape:", content_matrix.shape)
joblib.dump(content_matrix, "../Dataset/Processed/content_matrix.npy")

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
games_in_reviews = df_reviews["ID"].unique()
print("Games in reviews sample:", len(games_in_reviews))

from collections import defaultdict
game_texts = defaultdict(list)
for _, row in df_reviews.iterrows():
    gid = int(row["ID"])
    if len(game_texts[gid]) < 50:
        game_texts[gid].append(str(row["comment"]))

game_ids = []
embeddings = []
for gid, texts in game_texts.items():
    emb = model.encode(texts, show_progress_bar=False)
    mean_emb = emb.mean(axis=0)
    game_ids.append(gid)
    embeddings.append(mean_emb)

emb_df = pd.DataFrame(embeddings, index=game_ids)
emb_df.index.name = "id"
emb_df.to_parquet("../Dataset/Processed/game_review_embeddings.parquet")
print("Saved per-game embeddings:", emb_df.shape)

In [ ]:
from scipy.sparse import load_npz, csr_matrix
desc_tfidf = joblib.load("../Dataset/Processed/desc_tfidf.npz")
desc_sim = cosine_similarity(desc_tfidf, desc_tfidf)
joblib.dump(desc_sim, "../Dataset/Processed/desc_sim.npy")

content_sim = cosine_similarity(content_matrix, content_matrix)
joblib.dump(content_sim, "../Dataset/Processed/content_sim.npy")

emb = pd.read_parquet("../Dataset/Processed/game_review_embeddings.parquet")
emb_sim = cosine_similarity(emb.values, emb.values)
joblib.dump(emb_sim, "../Dataset/Processed/emb_sim.npy")

print("Saved similarity matrices")